# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abood-arc/Flyrank-ml-project/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [6]:
print("""Here's the rule, plainly: if a page's sitting at a decent spot in search but its click-through
rate is nowhere near what other pages at that spot usually pull in, it's worth a look — and it
matters more if that page happens to be carrying a big chunk of its own client's traffic.

Before I let myself trust that idea, I checked the two signals it's actually built on. Good
thing too.

**Signal 1 — does CTR actually drop as position gets worse?** That's the assumption sitting
under FlyRank's CTR-fix logic, so it's my flag-linked pick.

**Signal 2 — does staleness (how long since a page got touched) predict it doing badly?** That's
the assumption under the refresh flags.

Two calls I made that a reviewer might raise an eyebrow at:

- I used `content_updated_date`, not `last_optimized_date` — tempting to grab the one that
  literally has "optimized" in the name, but our own data contract already benched it. It tells
  you when FlyRank *decided* to work on a page, not anything about the page itself.
  `content_updated_date` is just the page's own edit history, no product-decision baggage.
- There's no decline label anywhere in this notebook, even though I'm supposedly in the
  Momentum/Recovery lane. Not an oversight — I don't have a validated window for one yet (that's
  `w05`'s job, per `work/CLAUDE.md`), and cobbling one together here risked the exact kind of
  leakage the ML-04 notebook already got burned by once. Everything below is just this window's
  current numbers: position, CTR, last-updated date. Nothing about the future.
.""")

Here's the rule, plainly: if a page's sitting at a decent spot in search but its click-through
rate is nowhere near what other pages at that spot usually pull in, it's worth a look — and it
matters more if that page happens to be carrying a big chunk of its own client's traffic.

Before I let myself trust that idea, I checked the two signals it's actually built on. Good
thing too.

**Signal 1 — does CTR actually drop as position gets worse?** That's the assumption sitting
under FlyRank's CTR-fix logic, so it's my flag-linked pick.

**Signal 2 — does staleness (how long since a page got touched) predict it doing badly?** That's
the assumption under the refresh flags.

Two calls I made that a reviewer might raise an eyebrow at:

- I used `content_updated_date`, not `last_optimized_date` — tempting to grab the one that
  literally has "optimized" in the name, but our own data contract already benched it. It tells
  you when FlyRank *decided* to work on a page, not anything about the page 

In [11]:
import os
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("SET enable_progress_bar=false")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
}

current_state_sql = f"""
    SELECT content_hash_id,
           ANY_VALUE(client_hash_id) AS client_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           SUM(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position * gsc_impressions ELSE 0 END)
             / NULLIF(SUM(CASE WHEN gsc_avg_position > 0 THEN gsc_impressions ELSE 0 END), 0) AS avg_position
    FROM {TABLES['fact_daily_sample']}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= 100
       AND SUM(CASE WHEN gsc_avg_position > 0 THEN gsc_impressions ELSE 0 END) > 0
"""
current_state = con.sql(current_state_sql).df()
print(f'{len(current_state):,} content items pass the evidence floor')  # expect 101,916

def position_bucket(p):
    if p <= 3: return '1_pos_1-3'
    if p <= 6: return '2_pos_4-6'
    if p <= 10: return '3_pos_7-10'
    if p <= 20: return '4_pos_11-20'
    return '5_pos_21plus'

current_state['position_bucket'] = current_state['avg_position'].apply(position_bucket)

signal1 = (current_state.groupby('position_bucket')
           .agg(n_content_items=('content_hash_id', 'count'),
                total_impressions=('impressions', 'sum'),
                total_clicks=('clicks', 'sum'))
           .reset_index())
signal1['weighted_ctr_pct'] = (signal1['total_clicks'] * 100 / signal1['total_impressions']).round(3)
print(signal1.to_string(index=False))

top_ctr = signal1.loc[signal1['position_bucket']=='1_pos_1-3','weighted_ctr_pct'].iloc[0]
rest_ctr = signal1.loc[signal1['position_bucket']!='1_pos_1-3','weighted_ctr_pct']
print(f"VERDICT: MIXED. Position 1-3 dominates (~{top_ctr:.1f}% CTR) but CTR does not decay "
      f"smoothly after that -- roughly flat {rest_ctr.min():.1f}-{rest_ctr.max():.1f}% from "
      f"position 4 to 20.")


dim_content = con.sql(f"SELECT content_hash_id, content_updated_date FROM {TABLES['dim_content']}").df()
staged = current_state.merge(dim_content, on='content_hash_id', how='left')
staged['staleness_days'] = (pd.Timestamp('2026-06-30') - pd.to_datetime(staged['content_updated_date'])).dt.days

def staleness_bucket(d):
    if d < 90: return '1_under_90d'
    if d < 180: return '2_90-179d'
    if d < 365: return '3_180-364d'
    return '4_365d_plus'

staged['staleness_bucket'] = staged['staleness_days'].apply(staleness_bucket)
signal2 = staged.groupby('staleness_bucket').agg(n=('content_hash_id', 'count'),
                                                    avg_staleness_days=('staleness_days', 'mean')).reset_index()
print(signal2.to_string(index=False))

full_library = dim_content.copy()
full_library['staleness_days'] = (pd.Timestamp('2026-06-30') - pd.to_datetime(full_library['content_updated_date'])).dt.days
full_library['staleness_bucket'] = full_library['staleness_days'].apply(staleness_bucket)
print(full_library['staleness_bucket'].value_counts(normalize=True).sort_index().round(3))

full_stale_pct = full_library['staleness_bucket'].value_counts(normalize=True).get('4_365d_plus', 0) * 100
active_stale_pct = staged['staleness_bucket'].value_counts(normalize=True).get('4_365d_plus', 0) * 100
print(f"VERDICT: FALSE as a predictor here. {full_stale_pct:.1f}% of the FULL library is 365d+ "
      f"stale, but among currently-ranking content that tail is {active_stale_pct:.1f}% -- stale "
      f"content mostly is not ranking well enough to even enter this population. Dropped from "
      f"the score; kept only as a cooldown below.")


101,916 content items pass the evidence floor
position_bucket  n_content_items  total_impressions  total_clicks  weighted_ctr_pct
      1_pos_1-3             2561          9132795.0      358111.0             3.921
      2_pos_4-6            15602         69215882.0      383629.0             0.554
     3_pos_7-10            28580         87394384.0      282697.0             0.323
    4_pos_11-20            25908         27031080.0      114204.0             0.422
   5_pos_21plus            29265         20914469.0       50211.0             0.240
VERDICT: MIXED. Position 1-3 dominates (~3.9% CTR) but CTR does not decay smoothly after that -- roughly flat 0.2-0.6% from position 4 to 20.
staleness_bucket     n  avg_staleness_days
     1_under_90d 89416           21.226246
       2_90-179d 12382          125.011872
      3_180-364d   118          228.567797
staleness_bucket
1_under_90d    0.737
2_90-179d      0.069
3_180-364d     0.058
4_365d_plus    0.136
Name: proportion, dtype: float64
VE

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Now signal 1 turns into an actual score. Signal 2 (staleness) didn't survive the check, so it's
not going into the score — but it comes back below in a different role: a cooldown filter.
"Staleness predicts underperformance" is what I tested and rejected. "Don't re-flag a page that
was just edited, give the fix time to show" is a separate, reasonable rule that doesn't depend
on the first claim being true at all.

Order matters here (straight from the lecture notes on eligibility-before-threshold): decide
who's even *allowed* to be scored, first — then score them. Backwards, and a 12-impression page
with a huge relative gap can outrank a page with real evidence behind it.

A heads-up on something a reviewer might not expect: my first version of this score was
`ctr_gap × raw impressions`. It looked fine until I checked *who* was showing up — 2 of 6
clients held 70% of the top 20, because it was really just ranking whoever has the most total
traffic. I switched to weighting by each page's *share of its own client's traffic* instead.
That fixed the big-client bias — but created a smaller one: a client with only 1 eligible page
trivially gets a 100% share. That's exactly why the ≥20-page client floor below exists — it's
not just borrowed from the label notebook for consistency, it's load-bearing here too.

In [12]:
# ============================================================
# The rule: eligibility gate -> client floor -> gap/expected CTR -> client share -> cooldown -> score
# ============================================================

# Client floor: a client with too few eligible pages doesn't have a meaningful "typical
# traffic" to weight against. Same floor, same reason, as the 20-page minimum
# w02_ml_task_framing.ipynb already uses for the label.
client_counts = current_state.groupby('client_hash_id')['content_hash_id'].count()
eligible_clients = client_counts[client_counts >= 20].index
scoped = current_state[current_state['client_hash_id'].isin(eligible_clients)].copy()
print(f'{len(scoped):,} items after client-size floor (dropped {len(current_state) - len(scoped):,} thin clients)')
# Expect: 101,877

# Expected CTR per position bucket, recomputed on this same client-floored population, keeps
# "expected" consistent with who's actually being scored.
expected_by_bucket = (scoped.groupby('position_bucket')
                       .apply(lambda g: g['clicks'].sum() * 100 / g['impressions'].sum(), include_groups=False))
scoped['item_ctr_pct'] = scoped['clicks'] * 100 / scoped['impressions']
scoped['expected_ctr_pct'] = scoped['position_bucket'].map(expected_by_bucket)
scoped['ctr_gap_pct'] = scoped['expected_ctr_pct'] - scoped['item_ctr_pct']

# Client traffic share, computed HERE, before cooldown removes anything. This ordering matters
# and if you compute this AFTER cooldown, a client can pass the 20-page
# floor easily, then lose most of those pages to the cooldown filter, leaving 2 or 3 pages
# splitting a tiny leftover denominator between them -- the exact small-denominator inflation
# the 20-page floor was supposed to prevent, sneaking back in through a different door. Computing
# it against the full client-floored portfolio, before cooldown thins it out, keeps "share"
# meaning "share of everything this client has," not "share of whatever happens to survive
# cooldown this run."
client_totals = scoped.groupby('client_hash_id')['impressions'].sum()
scoped['client_traffic_share'] = scoped['impressions'] / scoped['client_hash_id'].map(client_totals)

# Cooldown: exclude anything edited in the last 30 days. Signal 2 showed staleness doesn't
# PREDICT anything here -- but "give a just-edited page time before re-flagging it" is a
# separate, reasonable claim that doesn't depend on that being true.
scoped = scoped.merge(dim_content, on='content_hash_id', how='left')
scoped['staleness_days'] = (pd.Timestamp('2026-06-30') - pd.to_datetime(scoped['content_updated_date'])).dt.days
scoped = scoped[scoped['staleness_days'] >= 30].copy()
print(f'{len(scoped):,} items after cooldown -- this is the final scored pool')
# Expect: 47,171

# Score: gap x stake. Why not raw impressions: first version of this score used
# ctr_gap x raw impressions, and 2 of 6 distinct clients ended up holding 70% of the top 20 --
# it was really just ranking whoever has the most total traffic, not whoever has the worst
# relative problem. Client-share weighting fixes that.
scoped['score'] = (scoped['ctr_gap_pct'].clip(lower=0) * scoped['client_traffic_share']).round(4)

# One rule, one reason code, one action -- on purpose. This baseline tests one idea, not a
# tree of branching logic.
scoped['reason_code'] = 'ctr_below_position_peers'
scoped['action'] = 'flag_recovery_review'

ranked = scoped.sort_values('score', ascending=False).reset_index(drop=True)
ranked.insert(0, 'rank', ranked.index + 1)

output_cols = ['rank', 'content_hash_id', 'client_hash_id', 'position_bucket', 'avg_position',
               'impressions', 'clicks', 'item_ctr_pct', 'expected_ctr_pct', 'ctr_gap_pct',
               'client_traffic_share', 'score', 'reason_code', 'action']

import pathlib
pathlib.Path('work/outputs').mkdir(parents=True, exist_ok=True)
ranked[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f'wrote {len(ranked):,} ranked rows to work/outputs/baseline_action_score.csv')
ranked[output_cols].head(20)


101,877 items after client-size floor (dropped 39 thin clients)
47,171 items after cooldown -- this is the final scored pool
wrote 47,171 ranked rows to work/outputs/baseline_action_score.csv


,rank,content_hash_id,client_hash_id,position_bucket,avg_position,impressions,clicks,item_ctr_pct,expected_ctr_pct,ctr_gap_pct,client_traffic_share,score,reason_code,action
0,1,content_c1a10050d6a3c045,client_f623b01661d4bfe4,1_pos_1-3,2.450321,1248.0,3.0,0.240385,3.921154,3.680770,0.017437,0.0642,ctr_below_position_peers,flag_recovery_review
1,2,content_d36782b277dd9477,client_0b245132bb722950,2_pos_4-6,4.915943,9898.0,6.0,0.060618,0.554293,0.493674,0.123657,0.0610,ctr_below_position_peers,flag_recovery_review
2,3,content_cca099da6c658785,client_8ddc46da5414ffd8,1_pos_1-3,1.762946,157289.0,631.0,0.401172,3.921154,3.519982,0.013830,0.0487,ctr_below_position_peers,flag_recovery_review
3,4,content_eaccd7441ee37d02,client_a2eeb8899886adde,2_pos_4-6,5.004926,2639.0,1.0,0.037893,0.554293,0.516400,0.088465,0.0457,ctr_below_position_peers,flag_recovery_review
4,5,content_d7ea374befcbccd5,client_400c21c81c8b46ef,1_pos_1-3,2.450281,5330.0,43.0,0.806754,3.921154,3.114400,0.010229,0.0319,ctr_below_position_peers,flag_recovery_review
5,6,content_cffcd7471fdcfd0d,client_def0955f7a377868,3_pos_7-10,6.526874,5656.0,7.0,0.123762,0.323465,0.199703,0.152117,0.0304,ctr_below_position_peers,flag_recovery_review
6,7,content_dad2745a4507e9d1,client_65de48885f4ef01b,1_pos_1-3,2.627143,3441.0,24.0,0.697472,3.921154,3.223683,0.008799,0.0284,ctr_below_position_peers,flag_recovery_review
7,8,content_a9762b40c4d07aec,client_65de48885f4ef01b,3_pos_7-10,6.563194,33864.0,0.0,0.000000,0.323465,0.323465,0.086596,0.0280,ctr_below_position_peers,flag_recovery_review
8,9,content_094a85e7821fc587,client_ccdd78843409c8c7,3_pos_7-10,7.053425,2190.0,1.0,0.045662,0.323465,0.277803,0.081382,0.0226,ctr_below_position_peers,flag_recovery_review
9,10,content_05f68bef0a2d9f36,client_b10cb2997d0c7c86,3_pos_7-10,8.649814,26583.0,3.0,0.011285,0.323465,0.312180,0.072124,0.0225,ctr_below_position_peers,flag_recovery_review


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [13]:

print("""Rank one, this page sits at position 2.4, basically the best seat on the page, and still only pulls a 0.24% click rate against a 3.92% norm for that spot. Top of the list for a reason. What would prove me wrong is if this is a branded search where people already know where they're headed and don't bother clicking.

Rank two, good position, decent traffic, almost nobody clicks. Would be wrong if there's a near duplicate page on the same site quietly eating its clicks instead.

Rank three, the biggest volume miss on the whole list. Near number one position, over a hundred fifty thousand impressions, still miles under the norm. Would be wrong if 0.4% is just what a broad informational query like this one normally gets.

Rank four, one click on twenty six hundred impressions. Would be wrong if a hundred impressions just isn't enough evidence yet to trust that number.

Rank five, this one actually gets clicks, just about a third of what the norm expects. Would be wrong if the client's brand isn't established enough yet to earn trust clicks even from a good position.

Rank six, mid page one, a real gap but a smaller one. Would be wrong if the bucket's own expected CTR is itself dragged down by other broken pages sitting in the same bucket.

Rank seven, first of three pages from this same client showing up in this list. Worth checking whether it's one broken template across their whole site rather than three unrelated problems.

Rank eight, zero clicks on almost thirty four thousand impressions. The starkest number on the entire list. Would be wrong if this page only just started ranking here and simply hasn't had time to earn a click yet. Second appearance of the client from rank seven.

Rank nine, thin evidence again, one click on about two thousand impressions. Same caveat as rank four.

Rank ten, real volume, almost no clicks. Would be wrong if something technical broke midway through the month, a bad canonical tag or an accidental noindex, rather than this being a content problem at all.

Rank eleven, third appearance of that same client. This page actually earns real clicks, just fewer than its peers at the same position. Honestly the least broken pick on the list. Would be wrong if 0.16% is just normal variance for this client's niche.

Rank twelve, ranks so deep that even the expected CTR for its whole bucket is basically nothing. This pick barely stands out from thousands of similar deep pages, and that's its real weakness.

Rank thirteen, the best click rate of any position one to three page on this list, and it's still a third of the norm. Would be wrong if that 3.92% norm is itself inflated by a couple of viral outliers dragging the average up.

Rank fourteen, near number one position, almost nothing in clicks. One of the most extreme gaps here. Would be wrong if something is actually broken in how this page shows up in results, a missing title or description, rather than a content quality issue.

Rank fifteen, second appearance of the client from rank four. Thin evidence again, same caveat.

Rank sixteen, modest gap, modest volume. A lower confidence pick just by virtue of sitting near the bottom of this twenty.

Rank seventeen, real volume, a clear gap. Would be wrong if this page recently switched its target keyword and just hasn't settled into a new position yet.

Rank eighteen, third appearance of the client from rank four and fifteen. At this point that's three flagged pages from one account, which says more about that client than about any single page.

Rank nineteen, second appearance of the client from rank three. Same story, decent position, big audience, well under norm.

Rank twenty, the biggest audience in the bottom half of this list. The gap itself is smaller in absolute terms than the higher ranked rows, which is exactly why it landed here and not closer to the top.
One thing worth pointing out on its own instead of burying it in the line by line notes. Two different clients each show up three separate times in this top twenty. Might just be genuinely unlucky, three real problems landing on the same account by coincidence. Or it might be one broken thing, like a snippet template, quietly producing three symptoms instead of three separate causes. I can't tell which from the ranked list alone, so before anyone acts on those six rows as six independent findings, it's worth someone actually looking at that client's pages by hand first.
""")

Rank one, this page sits at position 2.4, basically the best seat on the page, and still only pulls a 0.24% click rate against a 3.92% norm for that spot. Top of the list for a reason. What would prove me wrong is if this is a branded search where people already know where they're headed and don't bother clicking.

Rank two, good position, decent traffic, almost nobody clicks. Would be wrong if there's a near duplicate page on the same site quietly eating its clicks instead.

Rank three, the biggest volume miss on the whole list. Near number one position, over a hundred fifty thousand impressions, still miles under the norm. Would be wrong if 0.4% is just what a broad informational query like this one normally gets.

Rank four, one click on twenty six hundred impressions. Would be wrong if a hundred impressions just isn't enough evidence yet to trust that number.

Rank five, this one actually gets clicks, just about a third of what the norm expects. Would be wrong if the client's brand

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [14]:
top20 = ranked[ranked['rank'] <= 20]
zero_click = top20[top20['clicks'] == 0]
weak_picks = zero_click.head(2) if len(zero_click) > 0 else top20.nsmallest(2, 'clicks')
print(weak_picks[output_cols].to_string(index=False))

print('''All inputs feeding the score are either current window observations or static content
facts known independent of any outcome. impressions, clicks, and avg_position are summed or
weighted over June 2026 only, nothing from beyond that window. content_updated_date is the
page's own edit history, known before and independent of this window's performance.
client_hash_id and content_hash_id are used only for grouping, the client floor and the
traffic share weighting, never fed into the score as a feature.

No decline or momentum label was built anywhere in this notebook. That is deliberate. The
feature window and label window design for that prediction is still an open question and it is real work for w05_model.ipynb, not this baseline. This rule only
answers whether a page's CTR is worse than its position.''')


 rank          content_hash_id          client_hash_id position_bucket  avg_position  impressions  clicks  item_ctr_pct  expected_ctr_pct  ctr_gap_pct  client_traffic_share  score              reason_code               action
    8 content_a9762b40c4d07aec client_65de48885f4ef01b      3_pos_7-10      6.563194      33864.0     0.0           0.0          0.323465     0.323465              0.086596 0.0280 ctr_below_position_peers flag_recovery_review
   12 content_65b8a4998e633d89 client_3197e6291363b4db    5_pos_21plus     81.546268      32279.0     0.0           0.0          0.240115     0.240115              0.083037 0.0199 ctr_below_position_peers flag_recovery_review
All inputs feeding the score are either current window observations or static content
facts known independent of any outcome. impressions, clicks, and avg_position are summed or
weighted over June 2026 only, nothing from beyond that window. content_updated_date is the
page's own edit history, known before and independent

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.